# FA2: GitHub Activity Analytics using PySpark — actual dataset

This notebook analyzes the downloaded actual GitHub event dataset. It uses PySpark for cleaning, deduplication, grouping, counting and ranking, then generates two bar charts, an advanced heatmap, CSV tables and a report.

**VS Code:** choose a Python 3.12 kernel and install Java 17+. Restart the kernel, then Run All. The first code cell configures a dedicated temporary directory before Spark starts; this may resolve the gateway temporary-file error. It does not guarantee a fix for OS-level locks or policies.

**Input:** `D:\SEM 7\FA2 BDA REAL DATA\GitHub_Actual_Events.jsonl`. Change DATA_PATH if needed. FETCH_FROM_GITHUB=False uses that existing actual dataset; True downloads new public events. No synthetic data is generated.

Output: `D:\SEM 7\FA2 BDA REAL DATA\github-real-results`.

In Colab change DATA_PATH to the uploaded file location and OUTPUT_DIR to a writable local folder. GitHub API events are a delayed sample, not a complete real-time stream.


## 1. Dependencies and runtime
The next cell installs missing Python libraries. Java must already be available. On a managed Windows computer, use an approved Java installation or a hosted notebook runtime.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Run after restarting the kernel, before Spark is created.
SPARK_TEMP = Path.home() / "spark-temp"
SPARK_TEMP.mkdir(parents=True, exist_ok=True)
os.environ["TEMP"] = str(SPARK_TEMP)
os.environ["TMP"] = str(SPARK_TEMP)
tempfile.tempdir = str(SPARK_TEMP)
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
with tempfile.TemporaryFile(mode="w+") as check:
    check.write("OK")
    check.seek(0)
    print("Temporary folder check:", check.read())
print("Python executable:", sys.executable)
print("Spark temporary folder:", SPARK_TEMP)
if sys.version_info >= (3, 14):
    print("Select a Python 3.12 kernel for the tested notebook environment.")

import importlib.util
import subprocess
import sys

missing = []
if importlib.util.find_spec("pyspark") is None:
    missing.append("pyspark==4.0.1")
if importlib.util.find_spec("matplotlib") is None:
    missing.append("matplotlib==3.9.4")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import shutil
if shutil.which("java") is None:
    raise RuntimeError("Java is missing. Use Java 17+ or a hosted runtime with Java installed.")
subprocess.run(["java", "-version"], check=True)
print("Python:", sys.version.split()[0])

## 2. Select the actual dataset
The default configuration reads your downloaded GitHub_Actual_Events.jsonl file. SOURCE_MODE must remain "real". Check that the file is at DATA_PATH. Enable FETCH_FROM_GITHUB only if you want the collection cell to append newly downloaded public events.


In [ ]:
from pathlib import Path

FETCH_FROM_GITHUB = False
DATA_PATH = Path(r"D:\SEM 7\FA2 BDA REAL DATA\GitHub_Actual_Events.jsonl")
OUTPUT_DIR = Path(r"D:\SEM 7\FA2 BDA REAL DATA\github-real-results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COLLECTION_ROUNDS = 1
PAGES_PER_ROUND = 3
POLL_SECONDS = 60
SOURCE_MODE = "real"
if COLLECTION_ROUNDS < 1 or not 1 <= PAGES_PER_ROUND <= 3 or POLL_SECONDS < 60:
    raise ValueError("Use positive rounds, 1-3 pages, and poll interval of at least 60 seconds.")


In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError


def fetch_public_events(page, etag=None):
    url = f"https://api.github.com/events?per_page=100&page={page}"
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "FA2-PySpark-Notebook",
               "X-GitHub-Api-Version": "2022-11-28"}
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
    if etag:
        headers["If-None-Match"] = etag
    try:
        with urlopen(Request(url, headers=headers), timeout=30) as response:
            records = json.load(response)
            if not isinstance(records, list):
                raise RuntimeError("Unexpected GitHub response: expected an event list.")
            return records, response.headers.get("ETag"), int(response.headers.get("X-Poll-Interval", "60"))
    except HTTPError as exc:
        if exc.code == 304:
            return [], etag, int(exc.headers.get("X-Poll-Interval", "60"))
        reset = exc.headers.get("X-RateLimit-Reset", "unknown")
        raise RuntimeError(f"GitHub HTTP {exc.code}. Rate-limit reset (Unix seconds): {reset}. Stop and retry later or use an optional GITHUB_TOKEN.") from None
    except URLError:
        raise RuntimeError("Cannot reach GitHub. Check internet access, then rerun this cell.") from None


INPUT_PATH = DATA_PATH
if FETCH_FROM_GITHUB:
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    seen = set()
    if DATA_PATH.exists():
        for line in DATA_PATH.read_text(encoding="utf-8").splitlines():
            if line.strip():
                seen.add(str(json.loads(line)["id"]))
    etags = {}
    new_count = 0
    try:
        with DATA_PATH.open("a", encoding="utf-8") as stream:
            for round_index in range(COLLECTION_ROUNDS):
                poll_interval = POLL_SECONDS
                round_count = 0
                for page in range(1, PAGES_PER_ROUND + 1):
                    events, etags[page], server_interval = fetch_public_events(page, etags.get(page))
                    poll_interval = max(poll_interval, server_interval)
                    for event in events:
                        event_id = str(event["id"])
                        if event_id not in seen:
                            event["source"] = "github-public-api"
                            event["collected_at"] = datetime.now(timezone.utc).isoformat()
                            stream.write(json.dumps(event) + "\n")
                            seen.add(event_id)
                            round_count += 1
                    stream.flush()
                new_count += round_count
                print(f"Round {round_index + 1}: {round_count} new actual events; saved unique IDs: {len(seen)}", flush=True)
                if round_index + 1 < COLLECTION_ROUNDS:
                    print(f"Waiting {poll_interval} seconds before the next round...", flush=True)
                    time.sleep(poll_interval)
    except KeyboardInterrupt:
        print("Collection interrupted; records already written remain available for analysis.")
    print("Dataset saved:", DATA_PATH.resolve())
else:
    if not DATA_PATH.is_file():
        raise FileNotFoundError(f"Actual input file not found: {DATA_PATH.resolve()}")
if not DATA_PATH.is_file() or DATA_PATH.stat().st_size == 0:
    raise ValueError("No events collected. Resolve the API error and rerun collection.")


## 3. Start Apache Spark and read the JSONL dataset
Spark runs locally with two threads. This demonstrates the Spark API, not a multi-machine cluster. Explicit nested schema reads `repo.name` and `actor.login`. UTC is used consistently.

In [ ]:
import os
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
from pyspark.sql import SparkSession, functions as F, types as T

SCHEMA = T.StructType([
    T.StructField("id", T.StringType()), T.StructField("type", T.StringType()),
    T.StructField("repo", T.StructType([T.StructField("name", T.StringType())])),
    T.StructField("actor", T.StructType([T.StructField("login", T.StringType())])),
    T.StructField("created_at", T.StringType()), T.StructField("source", T.StringType()),
    T.StructField("_corrupt_record", T.StringType()),
])

spark = (SparkSession.builder.appName("FA2-GitHub-Notebook").master("local[2]")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.local.dir", str(SPARK_TEMP / "spark-work")).getOrCreate())
spark.sparkContext.setLogLevel("WARN")
raw = spark.read.schema(SCHEMA).option("mode", "PERMISSIVE").json(INPUT_PATH.resolve().as_posix()).cache()
raw_count = raw.count()
print("Spark version:", spark.version, "| Input records:", raw_count)
raw.printSchema()

## 4. Clean data using Spark transformations
Extract fields with `select`, parse timestamps safely, `filter` invalid records, `dropDuplicates` by event ID, and derive date/hour columns. Cached DataFrames support repeated analysis. Counts are unique event counts, not numbers of commits.

In [ ]:
def clean_events(raw, source_mode="all"):
    flat = raw.select("id", F.col("type").alias("event_type"),
                      F.col("repo.name").alias("repository"),
                      F.col("actor.login").alias("developer"), "source", "_corrupt_record",
                      F.try_to_timestamp("created_at").alias("timestamp"))
    valid = F.col("_corrupt_record").isNull() & F.col("timestamp").isNotNull()
    for field in ["id", "event_type", "repository", "developer"]:
        valid = valid & F.col(field).isNotNull() & (F.length(F.trim(F.col(field))) > 0)
    cleaned = flat.filter(valid)
    synthetic = F.coalesce(F.col("source") == "synthetic-demo", F.lit(False)) | F.col("id").startswith("demo-") | F.col("id").startswith("synthetic-")
    cleaned = cleaned.filter(~synthetic)
    return cleaned.dropDuplicates(["id"]).withColumn("date", F.to_date("timestamp")).withColumn("hour", F.hour("timestamp"))

valid_all = clean_events(raw, "all").cache()
clean = clean_events(raw, SOURCE_MODE).cache()
n = clean.count()
if n == 0:
    raise ValueError("No actual events remain. Fetch from GitHub or select an actual event export.")
print("Valid unique events before source filter:", valid_all.count())
print("Selected unique events:", n)
clean.select("id", "event_type", "repository", "developer", "timestamp").show(10, truncate=False)

## 5. Analyze using Spark aggregations and actions
`groupBy().count()` calculates totals, `orderBy` ranks results, and `limit` retains top repositories/developers. `show`, `collect`, `first`, and `count` are actions. Only aggregated tables are collected for visualization.

In [ ]:
def aggregate(clean):
    return {
        "event_types": clean.groupBy("event_type").count().orderBy(F.desc("count"), "event_type"),
        "repositories": clean.groupBy("repository").count().orderBy(F.desc("count"), "repository").limit(10),
        "developers": clean.groupBy("developer").count().orderBy(F.desc("count"), "developer").limit(10),
        "activity_heatmap": clean.groupBy("date", "hour").count().orderBy("date", "hour"),
        "daily_activity": clean.groupBy("date").count().orderBy("date"),
    }

frames = aggregate(clean)
tables = {}
for name, frame in frames.items():
    print("\nSPARK RESULT:", name)
    frame.show(10, truncate=False)
    tables[name] = [row.asDict() for row in frame.collect()]

bounds = clean.agg(F.min("timestamp").alias("start"), F.max("timestamp").alias("end")).first()
synthetic_count = clean.filter(F.coalesce(F.col("source") == "synthetic-demo", F.lit(False)) | F.col("id").startswith("demo-") | F.col("id").startswith("synthetic-")).count()
assert synthetic_count == 0, "Synthetic records must not enter the actual-data results."
label = "ACTUAL PUBLIC GITHUB EVENTS (collected sample)"
summary = {"dataset_label": label, "input": str(INPUT_PATH), "spark_version": spark.version,
           "input_records": raw_count, "valid_unique_before_source_filter": valid_all.count(),
           "unique_events": n, "unique_repositories": clean.select("repository").distinct().count(),
           "unique_developers": clean.select("developer").distinct().count(), "synthetic_events": synthetic_count,
           "start_utc": str(bounds.start), "end_utc": str(bounds.end), "source_filter": SOURCE_MODE}
print(json.dumps(summary, indent=2))

## 6. Create three visualizations
1. Event-type bar chart: dominant types of observed activity.
2. Repository bar chart: most active repositories in the input sample.
3. **Advanced heatmap:** activity by date and hour in UTC.

Matplotlib renders Spark aggregates; it does not perform the data analysis. Zero heatmap cells indicate no observed events. A short real collection may yield a one-row heatmap; collect multiple days for useful temporal interpretation.

In [ ]:
def charts(tables, folder, label):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
    for key, column, title, filename in [
        ("event_types", "event_type", "GitHub activity by event type", "01_event_types.png"),
        ("repositories", "repository", "Most active repositories in the sample", "02_repositories.png"),
    ]:
        rows = list(reversed(tables[key]))
        fig, ax = plt.subplots(figsize=(11, 6), layout="constrained")
        bars = ax.barh([r[column] for r in rows], [r["count"] for r in rows], color="#2879ba")
        ax.bar_label(bars, padding=4)
        ax.set_xlim(0, max(r["count"] for r in rows) * 1.18)
        ax.set_xlabel("Number of unique events")
        ax.set_title(title + "\n" + label, loc="left", pad=15)
        fig.savefig(folder / filename, dpi=160)
        plt.close(fig)
    dates = [str(r["date"]) for r in tables["daily_activity"]]
    grid = [[0] * 24 for _ in dates]
    index = {date: i for i, date in enumerate(dates)}
    for row in tables["activity_heatmap"]:
        grid[index[str(row["date"])]][row["hour"]] = row["count"]
    # All dates in the observed window, including days with zero observed events.
    from datetime import date, timedelta
    first, last = date.fromisoformat(dates[0]), date.fromisoformat(dates[-1])
    count_days = (last - first).days + 1
    if count_days > 366:
        raise ValueError("Heatmap supports up to 366 days; analyze a smaller date range.")
    all_dates = [str(first + timedelta(days=i)) for i in range(count_days)]
    grid = [grid[index[d]] if d in index else [0] * 24 for d in all_dates]
    fig, ax = plt.subplots(figsize=(12, min(18, max(4, count_days * .32))), layout="constrained")
    image = ax.imshow(grid, aspect="auto", cmap="YlGnBu", interpolation="nearest")
    ax.set_xticks(range(24))
    tick_step = max(1, count_days // 30)
    ax.set_yticks(range(0, count_days, tick_step), all_dates[::tick_step])
    ax.set_xlabel("Hour of day (UTC)")
    ax.set_ylabel("Date (UTC)")
    ax.set_title("Daily and hourly activity heatmap\n" + label, loc="left", pad=15)
    fig.colorbar(image, ax=ax, label="Unique events")
    fig.savefig(folder / "03_heatmap.png", dpi=160)
    plt.close(fig)

charts(tables, OUTPUT_DIR, label)
from IPython.display import display, Image
for filename in ["01_event_types.png", "02_repositories.png", "03_heatmap.png"]:
    display(Image(filename=str(OUTPUT_DIR / filename)))

## 7. Data-derived insights and report
These findings use the actual records selected by Spark. Explain the collection window and sample limitations. A high event count does not demonstrate code quality or developer productivity. Push counts are not counts of individual commits.


In [ ]:
import csv
import html

def export_csv(path, rows):
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def report(summary, tables, folder):
    n = summary["unique_events"]
    event = tables["event_types"][0]
    repo = tables["repositories"][0]
    peak = max(tables["activity_heatmap"], key=lambda r: r["count"])
    findings = [
        f"{event['event_type']} is the largest event category: {event['count']} events ({event['count']/n:.1%} of this sample).",
        f"{repo['repository']} is the most active observed repository: {repo['count']} events ({repo['count']/n:.1%}).",
        f"The busiest observed date/hour is {peak['date']} at {peak['hour']:02d}:00 UTC, with {peak['count']} events.",
        "Recommendation: use repository activity as a first screening signal, then inspect event content and review backlogs before assigning support.",
        "Recommendation: collect several full days before making scheduling recommendations from the heatmap.",
    ]
    caveats = [
        "Results describe the input sample, not all GitHub activity or developer productivity.",
        "PushEvent counts measure pushes, not individual commits. WatchEvent usually represents a star action, not a page view.",
        "Polling the latest public events page may miss events and can be delayed; this is not a complete census.",
        "A zero heatmap cell means no event in the dataset, not proof that no activity occurred on GitHub.",
        "A short collection cannot establish typical daily/weekly GitHub activity patterns.",
    ]
    (folder / "insights.txt").write_text(summary["dataset_label"] + "\n\n" + "\n".join(findings) + "\n\nLimitations:\n" + "\n".join(caveats), encoding="utf-8")
    esc = html.escape
    body = """<!doctype html><html><head><meta charset='utf-8'><title>GitHub FA2 analysis</title>
<style>body{font:16px system-ui;max-width:1050px;margin:40px auto;padding:0 24px;color:#172438;background:#f5f7fa}section{background:white;padding:24px;margin:20px 0;border-radius:12px}img{width:100%;height:auto}pre{white-space:pre-wrap}li{margin:12px 0}</style></head><body><h1>GitHub Activity Analytics</h1><p>Apache Spark / PySpark · FA2</p>"""
    body += "<section><h2>Dataset and cleaning</h2><pre>" + esc(json.dumps(summary, indent=2, default=str)) + "</pre></section>"
    for name, title in [("01_event_types.png", "Event categories"), ("02_repositories.png", "Repository activity"), ("03_heatmap.png", "Advanced visualization: heatmap")]:
        body += f"<section><h2>{esc(title)}</h2><img src='{name}' alt='{esc(title)}'></section>"
    body += "<section><h2>Insights and recommendations</h2><ul>" + "".join("<li>" + esc(x) + "</li>" for x in findings) + "</ul><h2>Limitations</h2><ul>" + "".join("<li>" + esc(x) + "</li>" for x in caveats) + "</ul></section></body></html>"
    (folder / "report.html").write_text(body, encoding="utf-8")

for name, rows in tables.items():
    export_csv(OUTPUT_DIR / (name + ".csv"), rows)
report(summary, tables, OUTPUT_DIR)
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print((OUTPUT_DIR / "insights.txt").read_text(encoding="utf-8"))
print("Report:", (OUTPUT_DIR / "report.html").resolve())

## 8. Verify results and stop Spark
Event-type, daily and hourly counts should sum to the selected event count. Top-10 repositories are a subset and need not sum to the full dataset.

In [ ]:
for key in ["event_types", "activity_heatmap", "daily_activity"]:
    assert sum(row["count"] for row in tables[key]) == n, key
assert synthetic_count == 0
print("Checks passed: counts reconcile and no synthetic events were analyzed.")
spark.stop()


## FA2 presentation checklist

| Rubric | Notebook evidence |
|---|---|
| Problem understanding (1 mark) | GitHub problem statement |
| Spark implementation (3 marks) | Reading, cleaning, grouping, ranking and actions |
| Analysis and results (2 marks) | Spark tables, counts and exported CSVs |
| Visualization (2 marks) | Two bar charts and advanced heatmap |
| Interpretation and recommendations (2 marks) | Computed findings with sample limitations |

For final submission, rerun using actual data and save the executed notebook. Keep the charts and report generated by that run. This file intentionally has no prefilled results. Run it to produce outputs from actual events.